#  Machine Learning Model Training & Evaluation

##  Objective
Train, evaluate, and compare Gradient Boosting (**XGBoost**, **LightGBM**), **Random Forest**, and **Ridge Regressor** models to predict daily state-level Aadhaar enrolments using engineered temporal, lag, rolling, and interaction features.

In [1]:
import os, glob, json, joblib, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

# 2021 projected census populations — kept for feature enrichment only
STATE_POPULATION = {
    "Andaman & Nicobar":    397_000,
    "Andhra Pradesh":     49_577_103,
    "Arunachal Pradesh":   1_570_458,
    "Assam":              34_293_000,
    "Bihar":             121_243_000,
    "Chandigarh":          1_158_000,
    "Chhattisgarh":       28_724_000,
    "Dadra & Nagar Haveli and Daman & Diu": 615_000,
    "Delhi":              20_667_656,
    "Goa":                 1_586_250,
    "Gujarat":            66_750_000,
    "Haryana":            28_204_000,
    "Himachal Pradesh":    7_503_000,
    "Jammu and Kashmir":  14_999_397,
    "Jharkhand":          36_480_000,
    "Karnataka":          66_165_000,
    "Kerala":             35_125_000,
    "Ladakh":                316_000,
    "Lakshadweep":            73_183,
    "Madhya Pradesh":     82_232_000,
    "Maharashtra":       123_144_000,
    "Manipur":             3_091_545,
    "Meghalaya":           3_366_710,
    "Mizoram":             1_239_244,
    "Nagaland":            2_157_059,
    "Odisha":             45_429_000,
    "Puducherry":          1_413_542,
    "Punjab":             30_141_373,
    "Rajasthan":          79_502_477,
    "Sikkim":                682_000,
    "Tamil Nadu":         77_841_000,
    "Telangana":          38_705_209,
    "Tripura":             4_169_794,
    "Uttar Pradesh":     231_502_578,
    "Uttarakhand":        11_250_858,
    "West Bengal":       100_896_618,
}
_MEDIAN_POP = int(np.median(list(STATE_POPULATION.values())))

STATE_ALIASES = {
    "andaman and nicobar islands": "Andaman & Nicobar",
    "andaman & nicobar islands": "Andaman & Nicobar",
    "a & n islands": "Andaman & Nicobar",
    "andhra pradesh": "Andhra Pradesh", "arunachal pradesh": "Arunachal Pradesh",
    "assam": "Assam", "bihar": "Bihar", "chandigarh": "Chandigarh",
    "chhattisgarh": "Chhattisgarh", "chhatisgarh": "Chhattisgarh",
    "dadra and nagar haveli": "Dadra & Nagar Haveli and Daman & Diu",
    "daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "dadra and nagar haveli and daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "delhi": "Delhi", "nct of delhi": "Delhi", "goa": "Goa", "gujarat": "Gujarat",
    "haryana": "Haryana", "himachal pradesh": "Himachal Pradesh",
    "jammu and kashmir": "Jammu and Kashmir", "jammu & kashmir": "Jammu and Kashmir",
    "jharkhand": "Jharkhand", "karnataka": "Karnataka", "kerala": "Kerala",
    "ladakh": "Ladakh", "lakshadweep": "Lakshadweep",
    "madhya pradesh": "Madhya Pradesh", "maharashtra": "Maharashtra",
    "manipur": "Manipur", "meghalaya": "Meghalaya", "mizoram": "Mizoram",
    "nagaland": "Nagaland", "odisha": "Odisha", "orissa": "Odisha",
    "puducherry": "Puducherry", "pondicherry": "Puducherry", "punjab": "Punjab",
    "rajasthan": "Rajasthan", "sikkim": "Sikkim", "tamil nadu": "Tamil Nadu",
    "telangana": "Telangana", "tripura": "Tripura", "uttar pradesh": "Uttar Pradesh",
    "uttarakhand": "Uttarakhand", "uttaranchal": "Uttarakhand", "west bengal": "West Bengal"
}

def _norm(val):
    if pd.isna(val): return float('nan')
    return STATE_ALIASES.get(str(val).strip().lower(), str(val).strip().title())

def load_raw(data_dir="."):
    def _read(pattern):
        files = sorted(glob.glob(os.path.join(data_dir, pattern), recursive=True))
        dfs = [pd.read_csv(f, dtype={"state": str, "district": str}) for f in files]
        if not dfs: return pd.DataFrame()
        df = pd.concat(dfs, ignore_index=True)
        df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
        df["norm_state"] = df["state"].apply(_norm)
        return df

    enrol = _read("api_data_aadhar_enrolment/**/*.csv")
    if not enrol.empty:
        enrol["total_enrolments"] = enrol[["age_0_5","age_5_17","age_18_greater"]].fillna(0).sum(axis=1)
        enrol = enrol.groupby(["date","norm_state"])[["age_0_5","age_5_17","age_18_greater","total_enrolments"]].sum().reset_index()

    demo = _read("api_data_aadhar_demographic/**/*.csv")
    if not demo.empty:
        demo["demo_total"] = demo[["demo_age_5_17","demo_age_17_"]].fillna(0).sum(axis=1)
        demo = demo.groupby(["date","norm_state"])[["demo_age_5_17","demo_age_17_","demo_total"]].sum().reset_index()

    bio = _read("api_data_aadhar_biometric/**/*.csv")
    if not bio.empty:
        bio["bio_total"] = bio[["bio_age_5_17","bio_age_17_"]].fillna(0).sum(axis=1)
        bio = bio.groupby(["date","norm_state"])[["bio_age_5_17","bio_age_17_","bio_total"]].sum().reset_index()

    m = pd.merge(enrol, demo, on=["date","norm_state"], how="outer") if not enrol.empty else demo
    if not bio.empty: m = pd.merge(m, bio, on=["date","norm_state"], how="outer")
    for c in ["total_enrolments","demo_total","bio_total"]:
        if c in m.columns: m[c] = m[c].fillna(0)
    return m

def build_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["norm_state","date"]).reset_index(drop=True)

    states = df["norm_state"].dropna().unique()
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    grid = pd.MultiIndex.from_product([states, all_dates], names=["norm_state","date"]).to_frame(index=False)
    full = pd.merge(grid, df, on=["norm_state","date"], how="left")

    for c in ["age_0_5","age_5_17","age_18_greater","total_enrolments",
              "demo_age_5_17","demo_age_17_","demo_total","bio_age_5_17","bio_age_17_","bio_total"]:
        if c in full.columns: full[c] = full[c].fillna(0)

    # Calendar
    full["day_of_week"]      = full["date"].dt.dayofweek
    full["day_of_month"]     = full["date"].dt.day
    full["month"]            = full["date"].dt.month
    full["quarter"]          = full["date"].dt.quarter
    full["day_of_year"]      = full["date"].dt.dayofyear
    full["is_weekend"]       = full["day_of_week"].isin([5,6]).astype(int)
    full["days_since_start"] = (full["date"] - full["date"].min()).dt.days
    full["sin_dow"]   = np.sin(2*np.pi*full["day_of_week"]/7)
    full["cos_dow"]   = np.cos(2*np.pi*full["day_of_week"]/7)
    full["sin_month"] = np.sin(2*np.pi*full["month"]/12)
    full["cos_month"] = np.cos(2*np.pi*full["month"]/12)
    full["state_cat"] = full["norm_state"].astype("category").cat.codes

    # Population features — log-scaled, median fill for district rows not in lookup
    full["log_state_pop"] = np.log1p(
        full["norm_state"].map(STATE_POPULATION).fillna(_MEDIAN_POP))

    dfs = []
    for state, grp in full.groupby("norm_state"):
        grp = grp.sort_values("date").copy()
        pop = STATE_POPULATION.get(state, _MEDIAN_POP)

        for lag in [1, 7, 14, 30]:
            grp[f"lag_{lag}"]      = grp["total_enrolments"].shift(lag)
            grp[f"bio_lag_{lag}"]  = grp["bio_total"].shift(lag)
            grp[f"demo_lag_{lag}"] = grp["demo_total"].shift(lag)
        for w in [7, 14, 30]:
            s = grp["total_enrolments"].shift(1)
            grp[f"rolling_mean_{w}"] = s.rolling(w, min_periods=1).mean()
            grp[f"rolling_std_{w}"]  = s.rolling(w, min_periods=1).std().fillna(0)
            grp[f"bio_rolling_{w}"]  = grp["bio_total"].shift(1).rolling(w, min_periods=1).mean()
            grp[f"demo_rolling_{w}"] = grp["demo_total"].shift(1).rolling(w, min_periods=1).mean()
        grp["bio_to_enrol_ratio"]  = (grp["bio_rolling_7"]  / (grp["rolling_mean_7"] + 1)).fillna(0)
        grp["demo_to_enrol_ratio"] = (grp["demo_rolling_7"] / (grp["rolling_mean_7"] + 1)).fillna(0)

        # Per-capita AS FEATURES (not target) — gives cross-state normalization signal
        grp["enrol_per_1000"]          = grp["total_enrolments"] / pop * 1000
        grp["rolling_mean_7_per_1000"] = grp["rolling_mean_7"]   / pop * 1000
        grp["lag_1_per_1000"]          = grp["lag_1"]            / pop * 1000
        dfs.append(grp)

    return pd.concat(dfs, ignore_index=True).fillna(0)

raw_panel    = load_raw(data_dir=".")
processed_df = build_features(raw_panel)
print(f"Dataset shape: {processed_df.shape}")
print(f"States/districts: {processed_df['norm_state'].nunique()} | Date range: {processed_df['date'].min().date()} -> {processed_df['date'].max().date()}")
print(f"Absolute range check — UP: {processed_df[processed_df['norm_state']=='Uttar Pradesh']['total_enrolments'].mean():.0f}/day, "
      f"Lakshadweep: {processed_df[processed_df['norm_state']=='Lakshadweep']['total_enrolments'].mean():.1f}/day")
print(f"Per-1000 (feature): UP={processed_df[processed_df['norm_state']=='Uttar Pradesh']['enrol_per_1000'].mean():.4f}, "
      f"Lakshadweep={processed_df[processed_df['norm_state']=='Lakshadweep']['enrol_per_1000'].mean():.4f}")


Dataset shape: (15912, 54)
States/districts: 52 | Date range: 2025-03-01 -> 2025-12-31
Absolute range check — UP: 3329/day, Lakshadweep: 0.7/day
Per-1000 (feature): UP=0.0144, Lakshadweep=0.0091


## ⚙️ Model Training & Chronological Evaluation

We split data chronologically into an 80% historical training set and a 20% recent testing set.
We train XGBoost, LightGBM, Random Forest, and Ridge models and compare their metrics (R², RMSE, MAE).

In [2]:
# ── Training: Ridge/RF on raw+scaled, XGB/LGB on log1p; all enriched with pop features ──
target_col = "total_enrolments"
base_exclude = [
    "date","norm_state","state","district","pincode",
    "age_0_5","age_5_17","age_18_greater","total_enrolments",
    "demo_age_5_17","demo_age_17_","demo_total",
    "bio_age_5_17","bio_age_17_","bio_total",
    "enrol_per_1000",  # leakage: current-day target scaled
]

unique_dates = sorted(processed_df["date"].unique())
split_date   = unique_dates[int(len(unique_dates) * 0.8)]
train_df = processed_df[processed_df["date"] < split_date].copy()
test_df  = processed_df[processed_df["date"] >= split_date].copy()
print(f"Split: train up to {pd.Timestamp(split_date).date()} | Train: {len(train_df):,} | Test: {len(test_df):,}")

# State-level stats on train only (no leakage)
state_stats = (
    train_df.groupby("norm_state")[target_col]
    .agg(state_mean_enrol="mean", state_std_enrol="std", state_median_enrol="median").reset_index()
)
state_stats["state_std_enrol"] = state_stats["state_std_enrol"].fillna(0)
train_df = train_df.merge(state_stats, on="norm_state", how="left")
test_df  = test_df.merge(state_stats,  on="norm_state", how="left")
gm = train_df["state_mean_enrol"].mean()
for c in ["state_mean_enrol","state_std_enrol","state_median_enrol"]:
    test_df[c] = test_df[c].fillna(gm)
    train_df[c] = train_df[c].fillna(gm)

feature_cols = [c for c in train_df.columns if c not in base_exclude]
print(f"Feature count: {len(feature_cols)}")

X_tr = train_df[feature_cols];  X_te = test_df[feature_cols]
y_tr = train_df[target_col];    y_te = test_df[target_col]
y_tr_log = np.log1p(y_tr)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr);  X_te_s = sc.transform(X_te)

os.makedirs("pkl_models", exist_ok=True)
results = []
best_obj, best_r2 = None, -float("inf")

def fit_eval(name, model, Xtr, ytr, Xte, yte, fname, log_target=False):
    global best_obj, best_r2
    t0 = time.time()
    model.fit(Xtr, np.log1p(ytr) if log_target else ytr)
    el = round(time.time()-t0,2)
    if log_target:
        p_te = np.expm1(np.maximum(0, model.predict(Xte)))
        p_tr = np.expm1(np.maximum(0, model.predict(Xtr)))
    else:
        p_te = np.maximum(0, model.predict(Xte))
        p_tr = np.maximum(0, model.predict(Xtr))
    tr_r2 = r2_score(ytr, p_tr); te_r2 = r2_score(yte, p_te)
    te_rmse = float(np.sqrt(mean_squared_error(yte, p_te)))
    te_mae  = float(mean_absolute_error(yte, p_te))
    joblib.dump(model, f"pkl_models/{fname}")
    if te_r2 > best_r2: best_r2 = te_r2; best_obj = model
    results.append({"model":name,"train_r2":round(tr_r2,4),"test_r2":round(te_r2,4),
                    "train_rmse":None,"test_rmse":round(te_rmse,2),"test_mae":round(te_mae,2),
                    "test_mape":None,"training_seconds":el,"model_path":fname})
    tag = "[log1p]" if log_target else "[raw+scaled]"
    print(f"  {name:22s}  train_R2={tr_r2:.4f}  test_R2={te_r2:.4f}  RMSE={te_rmse:.0f}  [{el}s]  {tag}")

fit_eval("Ridge Baseline",    Ridge(alpha=1.0),
         X_tr_s, y_tr, X_te_s, y_te, "ridge_baseline_model.pkl", log_target=False)
fit_eval("Random Forest",     RandomForestRegressor(n_estimators=300,max_depth=6,min_samples_leaf=25,random_state=42,n_jobs=-1),
         X_tr_s, y_tr, X_te_s, y_te, "random_forest_model.pkl",  log_target=False)
fit_eval("XGBoost",           xgb.XGBRegressor(n_estimators=300,learning_rate=0.02,max_depth=4,subsample=0.8,
         colsample_bytree=0.8,min_child_weight=10,reg_alpha=0.1,reg_lambda=5.0,random_state=42,n_jobs=-1),
         X_tr, y_tr, X_te, y_te, "xgboost_model.pkl", log_target=True)
fit_eval("LightGBM",          lgb.LGBMRegressor(n_estimators=300,learning_rate=0.02,num_leaves=20,
         min_child_samples=20,subsample=0.8,colsample_bytree=0.8,reg_alpha=0.1,reg_lambda=5.0,random_state=42,verbose=-1),
         X_tr, y_tr, X_te, y_te, "lightgbm_model.pkl", log_target=True)

if best_obj: joblib.dump(best_obj, "pkl_models/best_model.pkl")

joblib.dump(sc, "pkl_models/raw_target_scaler.pkl")
joblib.dump(sc, "pkl_models/ridge_scaler.pkl")
state_stats.to_csv("pkl_models/state_stats.csv", index=False)
with open("pkl_models/feature_metadata.json","w") as f:
    json.dump({
        "feature_cols": feature_cols,
        "log_target": False,
        "raw_target_models": ["Ridge Baseline","Random Forest"],
        "log_target_models": ["XGBoost","LightGBM"],
        "split_date": str(pd.Timestamp(split_date).date())
    }, f, indent=2)

comp_path = "pkl_models/model_comparison.json"
existing  = json.load(open(comp_path)) if os.path.exists(comp_path) else []
lstm_only = [e for e in existing if e.get("model") == "LSTM (PyTorch)"]
with open(comp_path,"w") as f: json.dump(lstm_only + results, f, indent=2)

print()
print(pd.DataFrame(results).sort_values("test_r2",ascending=False)[
    ["model","train_r2","test_r2","test_rmse","test_mae"]].to_string(index=False))


Split: train up to 2025-10-31 | Train: 12,688 | Test: 3,224
Feature count: 44
  Ridge Baseline          train_R2=0.2570  test_R2=0.4176  RMSE=1395  [0.01s]  [raw+scaled]


  Random Forest           train_R2=0.3213  test_R2=0.1793  RMSE=1656  [0.94s]  [raw+scaled]


  XGBoost                 train_R2=0.5581  test_R2=0.2541  RMSE=1578  [0.52s]  [log1p]


  LightGBM                train_R2=0.6652  test_R2=0.2644  RMSE=1568  [2.76s]  [log1p]

         model  train_r2  test_r2  test_rmse  test_mae
Ridge Baseline    0.2570   0.4176    1394.82    577.38
      LightGBM    0.6652   0.2644    1567.57    578.73
       XGBoost    0.5581   0.2541    1578.41    543.34
 Random Forest    0.3213   0.1793    1655.73    605.41
